## AuxTel bias tests
Testing impact of ion pump on banding noise.

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u

In [2]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [4]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)
#await asyncio.gather(latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.01 sec
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, algorithm)
Read 6 history items for RemoteEvent(ATDomeTrajectory, 0, appliedSettingsMatchStart)
Read 100 history items for RemoteEvent(ATDomeTrajectory, 0, heartbeat)
Read 22 history items for RemoteEvent(ATDomeTrajectory, 0, logMessage)
Read 1 history items for RemoteEvent(ATDomeTrajectory, 0, settingsApplied)
Read 18 history items for RemoteEvent(ATDomeTrajectory, 0, summaryState)
Read historical data in 0.04 sec
Read 100 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Read 100 history items for RemoteEvent(ATHeaderServ

[[None, None, None, None, None, None, None], [None, None, None, None]]

trajectory DDS read queue is filling: 31 of 100 elements
mountStatus DDS read queue is full (100 elements); data may be lost
torqueDemand DDS read queue is filling: 31 of 100 elements
mountPositions DDS read queue is filling: 23 of 100 elements
nasymth_m3_mountMotorEncoders DDS read queue is filling: 31 of 100 elements
guidingAndOffsets DDS read queue is full (100 elements); data may be lost
currentTargetStatus DDS read queue is full (100 elements); data may be lost
mount_Nasmyth_Encoders DDS read queue is filling: 32 of 100 elements
mount_AzEl_Encoders DDS read queue is filling: 32 of 100 elements
mount_AzEl_Encoders python read queue is filling: 31 of 100 elements
measuredTorque DDS read queue is filling: 32 of 100 elements
measuredMotorVelocity DDS read queue is filling: 32 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 32 of 100 elements


In [5]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.DISABLED, settingsToApply='current')

[<State.ENABLED: 2>, <State.DISABLED: 1>]

In [6]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.DISABLED)

[<State.ENABLED: 2>, <State.DISABLED: 1>]

logMessage DDS read queue is filling: 62 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements
logMessage DDS read queue is filling: 16 of 100 elements


In [ ]:
tmp = await latiss.rem.atcamera.evt_heartbeat.next(flush=True)
print(tmp)

In [ ]:
tmp = await latiss.rem.atcamera.evt_heartbeat.next(flush=True)
print(tmp)

In [ ]:
# This worked the first time
await latiss.rem.atcamera.cmd_enterControl.set_start(timeout=5)

In [ ]:
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)

In [ ]:
# enable components if required
#await atcs.enable()
await latiss.enable()

In [ ]:
tmp=await latiss.rem.atarchiver.cmd_start.set_start()
print(tmp)

In [ ]:
tmp = await latiss.rem.atarchiver.evt_heartbeat.next(flush=True)
print(tmp)

In [ ]:
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.ENABLED)

In [ ]:
# Everything now enabled 5:35PM

In [ ]:
# Run 1 bias as a test
await latiss.take_bias(1)

In [ ]:
# 5 bias frames
for i in range(5):
    await latiss.take_bias(1)
    await asyncio.sleep(2)

In [ ]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

In [ ]:
# Everything back in standby, ecept ATPTg is DISABLED